In [75]:
import os
import re
import pandas as pd
from matplotlib import pyplot as plt
from tbparse import SummaryReader
from tueplots import bundles

# Keep plotting style consistent with existing slides
plt.rcParams.update(bundles.beamer_moml())
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'figure.dpi': 200})

# change directory to project root
os.chdir(os.path.expanduser("~/Desktop/pomdp_coder"))
print(os.getcwd())

C:\Users\Frederik\Desktop\pomdp_coder


In [76]:
base_dir = os.path.join("outputs")

def load_avg_reward_df(path: str):
    """Return the Average Episode Reward scalars for a given experiment directory."""
    reader = SummaryReader(path, extra_columns={'dir_name'})
    df = reader.scalars
    return df[df['tag'] == "Average Episode Reward"].reset_index(drop=True)

def get_clean_df(path: str):
    """Return a cleaned DataFrame with only relevant columns."""
    df = load_avg_reward_df(path)
    df['environment'] = path.split("\\")[-2]
    df['approach'] = path.split("\\")[-1]
    df['seed'] = df['dir_name'].apply(lambda x: re.search(r"_seed(\d+)", x).group(1))
    df = df.drop(columns=['tag', 'step', 'dir_name'])
    df = df[['environment', 'approach', 'seed', 'value']]
    return df

In [77]:
# add "four_rooms", later, not finished yet
envs = ["tiger", "rocksample", "empty", "corners", "lava", "unlock"]
methods = ["hardcoded", "ours", "tabular", "random"]

directories = {
    env: {
        method: os.path.join(base_dir, env, method)
        for method in methods
    }
    for env in envs
}
directories

{'tiger': {'hardcoded': 'outputs\\tiger\\hardcoded',
  'ours': 'outputs\\tiger\\ours',
  'tabular': 'outputs\\tiger\\tabular',
  'random': 'outputs\\tiger\\random'},
 'rocksample': {'hardcoded': 'outputs\\rocksample\\hardcoded',
  'ours': 'outputs\\rocksample\\ours',
  'tabular': 'outputs\\rocksample\\tabular',
  'random': 'outputs\\rocksample\\random'},
 'empty': {'hardcoded': 'outputs\\empty\\hardcoded',
  'ours': 'outputs\\empty\\ours',
  'tabular': 'outputs\\empty\\tabular',
  'random': 'outputs\\empty\\random'},
 'corners': {'hardcoded': 'outputs\\corners\\hardcoded',
  'ours': 'outputs\\corners\\ours',
  'tabular': 'outputs\\corners\\tabular',
  'random': 'outputs\\corners\\random'},
 'lava': {'hardcoded': 'outputs\\lava\\hardcoded',
  'ours': 'outputs\\lava\\ours',
  'tabular': 'outputs\\lava\\tabular',
  'random': 'outputs\\lava\\random'},
 'unlock': {'hardcoded': 'outputs\\unlock\\hardcoded',
  'ours': 'outputs\\unlock\\ours',
  'tabular': 'outputs\\unlock\\tabular',
  'random

In [78]:
dfs = []

for env, method in directories.items():
    for approach, dir_path in method.items():
        df = get_clean_df(dir_path)
        dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

In [79]:
# some averages did not get logged, so we add them manually; run only once!
missing_rows = pd.DataFrame([
    {"environment": "lava", "approach": "hardcoded", "seed": 1, "value": 0.06951353308570327},
    {"environment": "lava", "approach": "hardcoded", "seed": 4, "value": 0.12406196502394695},
    {"environment": "lava", "approach": "hardcoded", "seed": 9, "value": 0.25828055561004276},
])

final_df = pd.concat([final_df, missing_rows], ignore_index=True)

In [80]:
# check if wehave 40 for each environment (10 seeds x 4 methods)
final_df["environment"].value_counts()

environment
tiger         40
empty         40
corners       40
lava          40
unlock        40
rocksample    39
Name: count, dtype: int64